In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown, HTML

# Απενεργοποίηση του auto-scroll στην έξοδο του notebook
display(HTML("<style>.output_scroll { height: unset !important; }</style>"))

def simulate_dac(method, freq_s, num_cycles):
    clear_output(wait=True)
    
    # 1. Continuous (Original) Signal
    t_max = num_cycles * (1.0 / 300.0) # based on base frequency
    t_cont = np.linspace(0, t_max, 1000)
    x_cont = 0.6 * np.sin(2 * np.pi * 300 * t_cont) + 0.4 * np.cos(2 * np.pi * 700 * t_cont)
    x_cont = x_cont / np.max(np.abs(x_cont))

    # 2. Discrete Samples
    T_s = 1.0 / freq_s
    t_samp = np.arange(0, t_max + T_s/10, T_s)
    x_samp = 0.6 * np.sin(2 * np.pi * 300 * t_samp) + 0.4 * np.cos(2 * np.pi * 700 * t_samp)
    x_samp = x_samp / np.max(np.abs(x_samp))

    # 3. Reconstruction Methods
    t_recon = np.linspace(0, t_max, 1500)
    x_recon = np.zeros_like(t_recon)

    if method == 'Zero-Order Hold (ZOH)':
        # Step function (staircase)
        for i, t in enumerate(t_recon):
            idx = np.searchsorted(t_samp, t, side='right') - 1
            if idx >= 0 and idx < len(x_samp):
                x_recon[i] = x_samp[idx]
            else:
                x_recon[i] = 0

    elif method == 'First-Order Hold (FOH)':
        # Linear interpolation between samples
        for i, t in enumerate(t_recon):
            idx = np.searchsorted(t_samp, t, side='right') - 1
            if idx >= 0 and idx < len(x_samp) - 1:
                t0, t1 = t_samp[idx], t_samp[idx+1]
                x0, x1 = x_samp[idx], x_samp[idx+1]
                x_recon[i] = x0 + (x1 - x0) * (t - t0) / (t1 - t0)
            elif idx >= len(x_samp) - 1:
                x_recon[i] = x_samp[-1]
            else:
                x_recon[i] = x_samp[0]

    elif method == 'Ideal Sinc (Whittaker-Shannon)':
        # Bandlimited reconstruction using sinc basis functions without warnings
        for n, t_n in enumerate(t_samp):
            arg = np.pi / T_s * (t_recon - t_n)
            with np.errstate(divide='ignore', invalid='ignore'):
                sinc_fn = np.where(arg == 0, 1.0, np.sin(arg) / arg)
            x_recon += x_samp[n] * sinc_fn

    # 4. Plotting Setup (Legend placed outside/below horizontally)
    fig, ax = plt.subplots(figsize=(12, 5.5))
    
    ax.plot(t_cont * 1e3, x_cont, 'b-', alpha=0.35, linewidth=2, label='Original Continuous Signal $x_a(t)$')
    ax.plot(t_samp * 1e3, x_samp, 'ro', markersize=6, label='Discrete Samples $x[n]$')
    ax.plot(t_recon * 1e3, x_recon, 'g-', linewidth=2, label=f'Reconstructed ({method})')
    
    ax.set_title(f'Digital-to-Analog Reconstruction via {method}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Time [ms]', fontsize=10)
    ax.set_ylabel('Amplitude [V]', fontsize=10)
    
    # Horizontal legend placed below the plot area
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, fontsize=9)
    ax.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

# Layout Management & User Guide
display(Markdown("""
### User Guide: D/A Conversion & Reconstruction
* **Reconstruction Method:** Switch between **ZOH** (staircase steps), **FOH** (linear segments), and **Ideal Sinc** (bandlimited Whittaker-Shannon interpolation).
* **Sampling Freq ($F_s$):** Adjust how densely the signal is sampled.
* **Observation:** Notice how ZOH introduces sharp high-frequency steps (requiring a low-pass filter), FOH smooths out corners via linear approximation, and Ideal Sinc closely recovers the original continuous waveform.
"""))

method_dropdown = widgets.Dropdown(
    options=['Zero-Order Hold (ZOH)', 'First-Order Hold (FOH)', 'Ideal Sinc (Whittaker-Shannon)'],
    value='Zero-Order Hold (ZOH)',
    description='Method:',
    style={'description_width': 'initial'}
)

freq_slider = widgets.IntSlider(
    value=2500, min=1000, max=8000, step=500, 
    description='Fs (Hz):', 
    style={'description_width': 'initial'}
)

ui = widgets.HBox([method_dropdown, freq_slider])
display(ui)

out = widgets.interactive_output(simulate_dac, {'method': method_dropdown, 'freq_s': freq_slider, 'num_cycles': widgets.fixed(3)})
display(out)